In [2]:
import pandas as pd
import numpy as np
import re
import glob

In [3]:
df=pd.read_csv("../raw_data/amazon_india_2019.csv")

In [235]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2019_00000001,2019-01-05,CUST_2019_00037860,PROD_001930,Fitbit Sports Watch Premium,Electronics,Smart Watch,Fitbit,29122.09,0.00,...,False,NaN,3.0,Delivered,1,2019,1,0.06,True,3.4
1,TXN_2019_00000002,2019-01-09,CUST_2019_00002481,PROD_001683,Apple Mi Pad 8GB RAM Silver,Electronics,Tablets,Apple,66396.45,0.00,...,False,NaN,3.5,Delivered,1,2019,1,0.40,True,4.5
2,TXN_2019_00000003,01/02/2019,CUST_2018_00031347,PROD_000460,Oppo F9 64GB Black,Electronics,Smartphones,Oppo,29966.28,0.00,...,False,NaN,5.0,Delivered,1,2019,1,0.16,True,3.3
3,TXN_2019_00000004,2019-01-22,CUST_2019_00024783,PROD_000633,Oppo Reno 64GB White,Electronics,Smartphones,Oppo,34225.21,19.68,...,True,Republic Day Sale,4.5,Delivered,1,2019,1,0.20,True,3.3
4,TXN_2019_00000005,2019-01-31,CUST_2019_00018471,PROD_000571,Xiaomi Mi A3 256GB Blue,Electronics,Smartphones,Xiaomi,27092.49,0.00,...,False,NaN,NaN,Returned,1,2019,1,0.25,True,4.5


In [236]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(9721), 121605)

In [237]:
df["delivery_charges"].describe()

count    111884.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: delivery_charges, dtype: float64

In [238]:
df.drop(columns=["delivery_charges"], inplace=True)

In [239]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

In [240]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121605 entries, 0 to 121604
Data columns (total 33 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          121605 non-null  object 
 1   order_date              121605 non-null  object 
 2   customer_id             121605 non-null  object 
 3   product_id              121605 non-null  object 
 4   product_name            121605 non-null  object 
 5   category                121605 non-null  object 
 6   subcategory             121605 non-null  object 
 7   brand                   121605 non-null  object 
 8   original_price_inr      121605 non-null  object 
 9   discount_percent        121605 non-null  float64
 10  discounted_price_inr    121605 non-null  float64
 11  quantity                121605 non-null  int64  
 12  subtotal_inr            121605 non-null  float64
 13  final_amount_inr        121605 non-null  float64
 14  customer_city       

In [241]:
df.shape

(121605, 33)

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [242]:
df["order_date"].head(20)

0     2019-01-05
1     2019-01-09
2     01/02/2019
3     2019-01-22
4     2019-01-31
5     2019-01-28
6     2019-01-06
7     2019-01-29
8     2019-01-04
9     2019-01-08
10    20/01/2019
11    2019-01-12
12    2019-01-14
13    2019-01-17
14    2019-01-17
15    2019-01-08
16    2019-01-29
17    2019-01-04
18    2019-01-05
19    2019-01-16
Name: order_date, dtype: object

In [243]:
df["order_date"]= (df["order_date"].str.replace("/","-", regex=False) 
                    .str.replace(" ", "", regex=False))

parts = df["order_date"].str.split("-", expand=True)
year_last = parts[2].str.len()==4
df.loc[year_last,"order_date"]= (parts[2]+"-"+parts[0]+"-"+parts[1])

parts = df["order_date"].str.split("-", expand=True)
mask= parts[1].astype(int)>12
df.loc[mask,"order_date"]= (parts[0]+"-"+parts[2]+"-"+parts[1])


In [244]:
df["order_date"]=pd.to_datetime(df["order_date"], errors="coerce")

In [245]:
df["order_date"].min(),df["order_date"].max()

(Timestamp('2019-01-01 00:00:00'), Timestamp('2019-12-31 00:00:00'))

In [246]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [247]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [248]:
df["original_price_inr"].unique()[:20]

array([ 29122.09,  66396.45,  29966.28,  34225.21,  27092.49,  51931.91,
        38452.01,  30678.27, 126403.28,  38970.56,  93779.3 ,       nan,
        35126.85, 130245.68, 102500.15, 100856.23,   5170.68, 114546.72,
       104006.  , 140016.12])

In [249]:
df["original_price_inr"].dtypes

dtype('float64')

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [250]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [251]:
df["customer_rating"].describe()

count    84812.000000
mean         4.314560
std          0.569945
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [252]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    28152
5.0    21820
4.0    21355
3.5     8535
3.0     4950
Name: count, dtype: int64

In [253]:
df["customer_rating"].isna().sum()

np.int64(36793)

In [254]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    36793
4.5    28152
5.0    21820
4.0    21355
3.5     8535
3.0     4950
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [255]:
df["customer_city"]= df["customer_city"].str.strip().str.lower()

In [256]:
df["customer_city"].unique()

array(['kolkata', 'bangalore', 'lucknow', 'kanpur', 'indore', 'nagpur',
       'delhi', 'mumbai', 'patna', 'jaipur', 'saharanpur', 'vadodara',
       'kochi', 'visakhapatnam', 'bareilly', 'moradabad', 'surat',
       'ahmedabad', 'chennai', 'hyderabad', 'bhubaneswar', 'chandigarh',
       'pune', 'ludhiana', 'coimbatore', 'gorakhpur', 'allahabad',
       'meerut', 'madras', 'aligarh', 'varanasi', 'new delhi', 'chenai',
       'mumba', 'calcutta', 'delhi ncr', 'bombay', 'bengaluru',
       'bengalore', 'banglore'], dtype=object)

In [257]:
city_map = {
    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "bengaluru": "bangalore",
    "bengalore": "bangalore",
    "banglore": "bangalore",

    "new delhi": "delhi",
    "delhi ncr": "delhi"
}

In [258]:
df["customer_city"]= df["customer_city"].replace(city_map)

In [259]:
df["customer_city"] = df["customer_city"].str.title()

In [260]:
df["customer_city"].value_counts().head()

customer_city
Mumbai       16754
Delhi        14642
Bangalore    11826
Chennai       9739
Kolkata       8099
Name: count, dtype: int64

In [261]:
df["customer_city"].unique()

array(['Kolkata', 'Bangalore', 'Lucknow', 'Kanpur', 'Indore', 'Nagpur',
       'Delhi', 'Mumbai', 'Patna', 'Jaipur', 'Saharanpur', 'Vadodara',
       'Kochi', 'Visakhapatnam', 'Bareilly', 'Moradabad', 'Surat',
       'Ahmedabad', 'Chennai', 'Hyderabad', 'Bhubaneswar', 'Chandigarh',
       'Pune', 'Ludhiana', 'Coimbatore', 'Gorakhpur', 'Allahabad',
       'Meerut', 'Aligarh', 'Varanasi'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [262]:
bool_candidates=[]
bool_values={"true", "false","y","n","yes", "no","0","1"}

for col in df.columns:
    vals= set(df[col].astype(str). str.lower().dropna().unique())
    if vals & bool_values:
        bool_candidates.append(col)
bool_candidates



['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [263]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [264]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
False            True               False               51267
                                    True                23971
True             True               False               16339
False            False              False               11496
True             True               True                 7554
False            False              True                 5518
True             False              False                3720
                                    True                 1740
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [4]:
df["category"].value_counts().head(20)

category
Electronics                  121525
Electronic                       22
ELECTRONICS                      20
Electronics & Accessories        19
Electronicss                     19
Name: count, dtype: int64

In [5]:
df["category"] = df["category"].str.strip().str.lower()


In [6]:
category_map = {
    "electronic": "electronics",
    "electronics": "electronics",
    "electronics & accessories": "electronics",
    "electronicss": "electronics"
}

In [7]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [8]:
df["category"].value_counts()


category
Electronics    121605
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [270]:
df["delivery_days"].unique()

array(['5', '1', 'Same Day', '3', '6', '4', '2', '7', '15', '-1',
       'Express', '1-2 days', '0'], dtype=object)

In [271]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()


In [272]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [273]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [274]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [275]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [276]:
df["delivery_days"].unique()

array([ 5.,  1.,  0.,  3.,  6.,  4.,  2.,  7., 15., nan])

In [277]:
df["delivery_days"].isnull().sum()

np.int64(719)

In [278]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [279]:
df["delivery_days"].describe()

count    121605.000000
mean          3.693911
std           1.689283
min           0.000000
25%           3.000000
50%           4.000000
75%           5.000000
max          15.000000
Name: delivery_days, dtype: float64

In [280]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [281]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [282]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [283]:
duplicates.shape

(1188, 33)

In [284]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
2,TXN_2019_00000003,2019-01-02,CUST_2018_00031347,PROD_000460,Oppo F9 64GB Black,Electronics,Smartphones,Oppo,29966.28,0.00,...,False,NaN,5.0,Delivered,1,2019,1,0.16,True,3.3
79,TXN_2019_00000080,2019-01-14,CUST_2019_00001183,PROD_000551,OnePlus OnePlus 7T 64GB White,Electronics,Smartphones,OnePlus,89100.77,0.00,...,False,NaN,NaN,Delivered,1,2019,1,0.18,True,3.7
228,TXN_2019_00000229,2019-01-31,CUST_2019_00015373,PROD_001900,Xiaomi Watch Deluxe,Electronics,Smart Watch,Xiaomi,38060.16,0.00,...,False,NaN,3.5,Delivered,1,2019,1,0.08,True,3.5
407,TXN_2019_00000408,2019-01-29,CUST_2015_00005830,PROD_000217,Apple iPhone 8 16GB White,Electronics,Smartphones,Apple,150208.43,0.00,...,False,NaN,4.5,Returned,1,2019,1,0.18,False,3.2
687,TXN_2019_00000688,2019-01-27,CUST_2019_00043335,PROD_001687,Apple Slate 8GB RAM Silver,Electronics,Tablets,Apple,64703.89,9.93,...,False,NaN,5.0,Delivered,1,2019,1,0.63,True,4.5


In [285]:
df.duplicated().sum()

np.int64(0)

In [286]:
df = df.drop_duplicates()

In [287]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
19205,TXN_2019_00019206,2019-03-05,CUST_2015_00000229,PROD_001743,Realme Galaxy Tab 8GB RAM Silver,Electronics,Tablets,Realme,59171.91,50.85,...,True,Holi Festival,NaN,Delivered,3,2019,1,0.67,True,4.0
121143,TXN_2019_00019206_DUP,2019-03-05,CUST_2015_00000229,PROD_001743,Realme Galaxy Tab 8GB RAM Silver,Electronics,Tablets,Realme,59171.91,50.85,...,True,Holi Festival,NaN,Delivered,3,2019,1,0.67,True,4.0
98940,TXN_2019_00098941,2019-11-27,CUST_2015_00001258,PROD_000512,Samsung Galaxy S10+ 64GB Black,Electronics,Smartphones,Samsung,88925.26,0.00,...,False,NaN,NaN,Delivered,11,2019,4,0.22,True,4.7
121157,TXN_2019_00098941_DUP,2019-11-27,CUST_2015_00001258,PROD_000512,Samsung Galaxy S10+ 64GB Black,Electronics,Smartphones,Samsung,88925.26,0.00,...,False,NaN,NaN,Delivered,11,2019,4,0.22,True,4.7
48292,TXN_2019_00048293,2019-06-30,CUST_2015_00001486,PROD_000287,Xiaomi Redmi Note 4 16GB White,Electronics,Smartphones,Xiaomi,29620.86,57.69,...,True,Back to School,4.5,Delivered,6,2019,2,0.25,False,3.5
121156,TXN_2019_00048293_DUP,2019-06-30,CUST_2015_00001486,PROD_000287,Xiaomi Redmi Note 4 16GB White,Electronics,Smartphones,Xiaomi,29620.86,57.69,...,True,Back to School,4.5,Delivered,6,2019,2,0.25,False,3.5
34797,TXN_2019_00034798,2019-05-12,CUST_2015_00002359,PROD_000069,Xiaomi Mi 4i 64GB White,Electronics,Smartphones,Xiaomi,24433.29,60.46,...,True,Summer Sale,3.5,Delivered,5,2019,2,0.23,False,3.6
121571,TXN_2019_00034798_DUP,2019-05-12,CUST_2015_00002359,PROD_000069,Xiaomi Mi 4i 64GB White,Electronics,Smartphones,Xiaomi,24433.29,60.46,...,True,Summer Sale,3.5,Delivered,5,2019,2,0.23,False,3.6
54793,TXN_2019_00054794,2019-07-30,CUST_2015_00002699,PROD_000528,Samsung Galaxy A50 128GB White,Electronics,Smartphones,Samsung,NaN,0.00,...,False,NaN,4.5,Delivered,7,2019,3,0.22,True,4.0
121245,TXN_2019_00054794_DUP,2019-07-30,CUST_2015_00002699,PROD_000528,Samsung Galaxy A50 128GB White,Electronics,Smartphones,Samsung,NaN,0.00,...,False,NaN,4.5,Delivered,7,2019,3,0.22,True,4.0


In [288]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000229  PROD_001743  2019-03-05  59171.91              2
CUST_2015_00001258  PROD_000512  2019-11-27  88925.26              2
CUST_2015_00001486  PROD_000287  2019-06-30  29620.86              2
CUST_2015_00002359  PROD_000069  2019-05-12  24433.29              2
CUST_2015_00002835  PROD_000612  2019-05-09  47065.25              2
CUST_2015_00003043  PROD_000162  2019-01-26  48896.17              2
CUST_2015_00003352  PROD_001856  2019-07-04  25055.85              2
CUST_2015_00005125  PROD_000316  2019-08-12  26457.79              2
CUST_2015_00005281  PROD_000318  2019-09-25  26068.49              2
CUST_2015_00005696  PROD_000637  2019-07-13  42858.35              2
dtype: int64

In [289]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [290]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [291]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2018_00031347,PROD_000460,1,2
1,CUST_2019_00001183,PROD_000551,1,2
2,CUST_2019_00015373,PROD_001900,1,2
3,CUST_2015_00005830,PROD_000217,2,2
4,CUST_2019_00043335,PROD_001687,1,2


In [292]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [293]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [294]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [295]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [296]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers (100x decimal error) ────────────
subcategory_caps = {
    "Smart Watch":        60000,
    "Tablets":            110000,
    "Smartphones":        250000,
    "Laptops":            240000,
    "TV & Entertainment": 300000,
    "Audio":              50000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 302
Outliers fixed: 1889
                         min        max       mean        50%
subcategory                                                  
Audio                 706.54   49445.00   20139.37   23006.17
Laptops              2401.31  235043.46  102307.69   90975.60
Smart Watch           600.39   58414.95   35941.80   35685.00
Smartphones          2531.81  246368.90   69124.87   45800.60
TV & Entertainment  11346.71  287334.40  126712.61  130245.68
Tablets              1149.64  105276.31   51924.00   57094.76

NaN in final_amount_inr:   3613
Negative prices remaining: 0


In [297]:
# Check the products exceeding caps
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}):")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))

In [298]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch: ✅ All within cap

Tablets: ✅ All within cap

Smartphones: ✅ All within cap



Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [299]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
COD            48696
UPI            24737
Credit Card    20571
Debit Card     19147
Net Banking     8454
Name: count, dtype: int64

NaN in payment_category: 0


Handling nan values in original_price_inr

In [300]:
print(f"NaN in original_price_inr: {df['original_price_inr'].isna().sum()}")

# Check the actual NaN rows
nan_rows = df[df["discounted_price_inr"].isna()]
print(nan_rows[["product_name", "original_price_inr", "discount_percent", "discounted_price_inr"]].head(10))

NaN in original_price_inr: 3613
                        product_name  original_price_inr  discount_percent  \
11        Apple iPhone XS 128GB Blue                 NaN             20.91   
12   Xiaomi Redmi Note 7 128GB Black                 NaN              0.00   
17    OnePlus OnePlus 6T 128GB White                 NaN             19.96   
129         MSI Aspire 4GB RAM Black                 NaN             18.56   
182       Realme Slate 4GB RAM Black                 NaN             15.13   
324    Samsung Galaxy A8+ 64GB Black                 NaN              0.00   
381          Xiaomi Mi 4i 64GB White                 NaN             64.15   
394       HP Inspiron 8GB RAM Silver                 NaN             11.36   
446   Samsung Galaxy S10 128GB White                 NaN             22.51   
496        Sennheiser Gaming Headset                 NaN             24.41   

     discounted_price_inr  
11                    NaN  
12                    NaN  
17                    NaN

In [301]:
# Check which subcategories these NaN prices belong to
print(df[df["original_price_inr"].isna()]["subcategory"].value_counts())

subcategory
Smartphones           2645
Laptops                315
Tablets                261
Smart Watch            206
Audio                  143
TV & Entertainment      43
Name: count, dtype: int64


In [302]:
# See the full distribution of Smartphones over cap
print(df[df["subcategory"] == "Smartphones"]["original_price_inr"]
      .describe().round(2))

count     86421.00
mean      69124.87
std       50721.80
min        2531.81
25%       31534.18
50%       45800.60
75%      102519.83
max      246368.90
Name: original_price_inr, dtype: float64


In [303]:
print(df[df["original_price_inr"].isna()].head(10).to_string())

        transaction_id order_date         customer_id   product_id                     product_name     category  subcategory       brand  original_price_inr  discount_percent  discounted_price_inr  quantity  subtotal_inr  final_amount_inr customer_city customer_state customer_tier customer_spending_tier customer_age_group payment_method  delivery_days delivery_type  is_prime_member  is_festival_sale      festival_name  customer_rating return_status  order_month  order_year  order_quarter  product_weight_kg  is_prime_eligible  product_rating  payment_category
11   TXN_2019_00000012 2019-01-12  CUST_2018_00020082  PROD_000348       Apple iPhone XS 128GB Blue  Electronics  Smartphones       Apple                 NaN             20.91                   NaN         2           NaN               NaN         Delhi          Delhi         Metro                Premium              46-55     Debit Card            2.0       Express             True             False                NaN            

In [304]:
# Drop rows with NaN original_price_inr
df = df.dropna(subset=["original_price_inr"]).copy()  # ← added .copy()

# Recalculate
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]




In [305]:
df["original_price_inr"].isna().sum()

np.int64(0)

Handling Nan - in customer age group

In [306]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [307]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [308]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")


category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
64.20022392272949 MB


In [309]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 117992

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        80389       68.13
customer_rating      35659       30.22


In [310]:
df.to_csv("data_cleaning_2019.csv", index=False)
print("File saved successfully!")

File saved successfully!
